# 第13章（二）：文件组织与访问 File Organisation and Access

## Cambridge A-Level Computer Science (9618)

---

### 本节学习目标 Learning Objectives

1. 理解三种文件组织方式：串行、顺序、随机/直接访问
2. 掌握哈希算法和哈希表的原理
3. 理解冲突处理的两种方法：开放寻址法和链地址法
4. 深入理解为什么哈希是计算机科学的核心技术

---

### 引言：为什么文件组织方式很重要？

想象你有一本1000页的电话簿：
- **串行 Serial**：按照信息到达的时间顺序排列 -> 找到"张伟"可能需要翻遍所有页
- **顺序 Sequential**：按姓名字母排序 -> 可以用二分查找，快得多
- **随机/直接 Random**：通过计算直接跳到"张伟"所在的页 -> 最快！

> 文件组织方式决定了**数据存储和检索的效率**，这在处理大量数据时至关重要。

In [ ]:
# 环境配置
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import random
import time
import os
import struct
import hashlib

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

print('环境配置完成！')

---

## 一、串行文件组织 Serial File Organisation

### 是什么？

数据按照**到达/输入的时间顺序**依次存储，**没有任何排序**。

### 特点
- 新记录添加到**文件末尾**
- 查找某条记录需要**从头到尾逐个检查**（线性搜索 Linear Search）
- 最坏情况下需要读取**所有记录**

### 为什么还有人用串行文件？

1. **写入速度最快**：直接追加到末尾，不需要维护顺序
2. **适合日志文件**：事件按时间顺序记录，很少需要搜索单条记录
3. **适合批量处理**：反正需要处理每条记录，顺序无所谓
4. **实现最简单**：不需要额外的索引或排序逻辑

In [ ]:
# === 串行文件组织 Serial File Demo ===

class SerialFile:
    """串行文件模拟"""
    def __init__(self):
        self.records = []
        self.access_count = 0
    
    def add_record(self, record):
        """添加到末尾 - O(1)"""
        self.records.append(record)
    
    def search(self, key_field, value):
        """线性搜索 - O(n)"""
        self.access_count = 0
        for record in self.records:
            self.access_count += 1
            if record[key_field] == value:
                return record, self.access_count
        return None, self.access_count

# 模拟日志系统
log_file = SerialFile()
events = [
    {"time": "09:15", "user": "Alice", "action": "LOGIN"},
    {"time": "09:20", "user": "Bob", "action": "LOGIN"},
    {"time": "09:25", "user": "Alice", "action": "VIEW_FILE"},
    {"time": "09:30", "user": "Charlie", "action": "LOGIN"},
    {"time": "09:35", "user": "Bob", "action": "EDIT_FILE"},
    {"time": "09:40", "user": "Alice", "action": "LOGOUT"},
    {"time": "09:45", "user": "Diana", "action": "LOGIN"},
    {"time": "09:50", "user": "Charlie", "action": "LOGOUT"},
]

print("=== 串行文件：系统日志 Serial File: System Log ===")
for e in events:
    log_file.add_record(e)
    print(f"  [{e['time']}] {e['user']} - {e['action']}")

print(f"\n记录总数: {len(log_file.records)}")

# 搜索
result, accesses = log_file.search("user", "Diana")
print(f"\n搜索 'Diana': 找到 {result}")
print(f"  需要访问 {accesses} 条记录（共 {len(log_file.records)} 条）")

result, accesses = log_file.search("user", "Alice")
print(f"\n搜索 'Alice': 找到 {result}")
print(f"  需要访问 {accesses} 条记录（找到第一个匹配就停止）")

---

## 二、顺序文件组织 Sequential File Organisation

### 是什么？

数据按照**某个关键字段（key field）的顺序**排列存储。

### 特点
- 记录按**键值排序**存储
- 可以使用**二分查找 (Binary Search)** 加速搜索
- 插入新记录**需要移动其他记录**以维持顺序
- 通常使用**溢出区/事务文件**暂存新记录，定期合并

### 为什么需要顺序文件？

1. **搜索效率高**：二分查找 O(log n) vs 线性搜索 O(n)
2. **适合范围查询**：如"查找所有成绩在80-90分之间的学生"
3. **适合批量顺序处理**：如按顺序打印工资单
4. **节省空间**：不需要额外的索引结构

In [ ]:
# === 顺序文件组织 Sequential File Demo ===

class SequentialFile:
    """顺序文件模拟"""
    def __init__(self, key_field):
        self.records = []
        self.key_field = key_field
        self.access_count = 0
    
    def add_record(self, record):
        """插入并保持有序 - O(n)"""
        self.records.append(record)
        self.records.sort(key=lambda r: r[self.key_field])
    
    def binary_search(self, value):
        """二分查找 - O(log n)"""
        self.access_count = 0
        low, high = 0, len(self.records) - 1
        while low <= high:
            mid = (low + high) // 2
            self.access_count += 1
            mid_val = self.records[mid][self.key_field]
            if mid_val == value:
                return self.records[mid], self.access_count
            elif mid_val < value:
                low = mid + 1
            else:
                high = mid - 1
        return None, self.access_count

# 创建顺序文件
seq_file = SequentialFile("id")
students = [
    {"id": "S005", "name": "Eve"},
    {"id": "S002", "name": "Bob"},
    {"id": "S008", "name": "Henry"},
    {"id": "S001", "name": "Alice"},
    {"id": "S003", "name": "Charlie"},
    {"id": "S007", "name": "Grace"},
    {"id": "S004", "name": "Diana"},
    {"id": "S006", "name": "Frank"},
]

print("=== 顺序文件：学生名册 Sequential File: Student Registry ===")
print("数据输入顺序（无序）:")
for s in students:
    print(f"  {s['id']}: {s['name']}")
    seq_file.add_record(s)

print("\n存储后（按ID排序）:")
for s in seq_file.records:
    print(f"  {s['id']}: {s['name']}")

# 二分查找
print("\n=== 二分查找演示 Binary Search ===")
for search_id in ["S003", "S007", "S009"]:
    result, accesses = seq_file.binary_search(search_id)
    if result:
        print(f"搜索 {search_id}: 找到 {result['name']}，访问了 {accesses} 次")
    else:
        print(f"搜索 {search_id}: 未找到，访问了 {accesses} 次")

print(f"\n对比：线性搜索最多需要 {len(seq_file.records)} 次")
print(f"       二分搜索最多需要 {int(np.ceil(np.log2(len(seq_file.records))))} 次")

---

## 三、随机/直接访问文件 Random (Direct Access) File Organisation

### 是什么？

通过**哈希函数 (hash function)** 将记录的键值映射到一个**存储地址**，从而可以**直接跳到该地址**读取数据，不需要搜索。

### 核心思想

```
存储地址 = hash(键值) MOD 文件大小
```

### 为什么随机访问文件如此重要？

1. **O(1) 访问时间**：不管有多少条记录，理想情况下只需**一次磁盘访问**
2. **数据库的基础**：几乎所有数据库都在底层使用哈希或B树
3. **操作系统的核心**：文件系统、页表、缓存都依赖哈希
4. **网络安全**：密码存储、数字签名、区块链都基于哈希

In [ ]:
# === 哈希函数演示 Hash Function Demo ===

def simple_hash(key, table_size):
    """简单哈希函数：将字符串的ASCII值求和后取模"""
    total = sum(ord(char) for char in str(key))
    return total % table_size

def better_hash(key, table_size):
    """改进的哈希函数：加入位置权重"""
    total = 0
    for i, char in enumerate(str(key)):
        total += ord(char) * (31 ** i)  # 31是常用质数
    return total % table_size

# 演示哈希计算过程
table_size = 11  # 使用质数作为表大小（减少碰撞）
keys = ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank", "Grace"]

print(f"=== 哈希函数演示 (表大小 = {table_size}) ===")
print(f"{'Key':<10} {'ASCII Sum':<12} {'Simple Hash':<14} {'Better Hash':<14}")
print("-" * 50)
for key in keys:
    ascii_sum = sum(ord(c) for c in key)
    h1 = simple_hash(key, table_size)
    h2 = better_hash(key, table_size)
    print(f"{key:<10} {ascii_sum:<12} {h1:<14} {h2:<14}")

print(f"\n注意：'Alice' 和 'Bob' 可能映射到同一位置 -> 这就是碰撞 Collision!")

In [ ]:
# === 哈希表实现 Hash Table Implementation ===

class HashTable:
    """哈希表 - 开放寻址法 Open Addressing (Linear Probing)"""
    
    def __init__(self, size=11):
        self.size = size
        self.table = [None] * size
        self.collision_count = 0
        self.probe_history = []  # 记录探查过程
    
    def _hash(self, key):
        return sum(ord(c) for c in str(key)) % self.size
    
    def insert(self, key, value):
        """插入：使用线性探查法处理碰撞"""
        index = self._hash(key)
        original_index = index
        probes = [index]
        
        while self.table[index] is not None:
            if self.table[index][0] == key:
                self.table[index] = (key, value)
                self.probe_history.append((key, probes, False))
                return
            self.collision_count += 1
            index = (index + 1) % self.size  # 线性探查
            probes.append(index)
            if index == original_index:
                raise Exception("Hash table is full!")
        
        self.table[index] = (key, value)
        self.probe_history.append((key, probes, len(probes) > 1))
    
    def search(self, key):
        """查找"""
        index = self._hash(key)
        original_index = index
        probes = 0
        
        while self.table[index] is not None:
            probes += 1
            if self.table[index][0] == key:
                return self.table[index][1], probes
            index = (index + 1) % self.size
            if index == original_index:
                break
        return None, probes
    
    def display(self):
        print(f"{'Index':<8} {'Content':<30}")
        print("-" * 38)
        for i, slot in enumerate(self.table):
            if slot:
                print(f"  [{i:>2}]   {slot[0]}: {slot[1]}")
            else:
                print(f"  [{i:>2}]   <empty>")

# 使用哈希表
ht = HashTable(11)
data = [
    ("Alice", "Computer Science"),
    ("Bob", "Mathematics"),
    ("Charlie", "Physics"),
    ("Diana", "Chemistry"),
    ("Eve", "Biology"),
    ("Frank", "English"),
    ("Grace", "History"),
]

print("=== 哈希表插入过程 Hash Table Insertion ===")
for key, value in data:
    ht.insert(key, value)

for key, probes, had_collision in ht.probe_history:
    collision_str = " ** COLLISION! **" if had_collision else ""
    print(f"  Insert '{key}': probed positions {probes}{collision_str}")

print(f"\n总碰撞次数: {ht.collision_count}")
print(f"\n=== 哈希表内容 Hash Table Contents ===")
ht.display()

# 搜索
print(f"\n=== 搜索演示 Search ===")
for name in ["Alice", "Grace", "Unknown"]:
    result, probes = ht.search(name)
    if result:
        print(f"  Search '{name}': Found '{result}' in {probes} probe(s)")
    else:
        print(f"  Search '{name}': Not found after {probes} probe(s)")

In [ ]:
# === 链地址法 Separate Chaining ===

class ChainedHashTable:
    """哈希表 - 链地址法 Separate Chaining"""
    
    def __init__(self, size=7):
        self.size = size
        self.table = [[] for _ in range(size)]  # 每个槽是一个链表
        self.collision_count = 0
    
    def _hash(self, key):
        return sum(ord(c) for c in str(key)) % self.size
    
    def insert(self, key, value):
        index = self._hash(key)
        # 检查是否已存在
        for i, (k, v) in enumerate(self.table[index]):
            if k == key:
                self.table[index][i] = (key, value)
                return
        if len(self.table[index]) > 0:
            self.collision_count += 1
        self.table[index].append((key, value))
    
    def display(self):
        for i, chain in enumerate(self.table):
            items = " -> ".join([f"({k}: {v})" for k, v in chain]) if chain else "<empty>"
            print(f"  [{i}]: {items}")

# 演示
cht = ChainedHashTable(7)
for key, value in data:
    cht.insert(key, value)

print("=== 链地址法哈希表 Chained Hash Table (size=7) ===")
cht.display()
print(f"\n碰撞次数: {cht.collision_count}")
print("\n注意：碰撞的元素形成链表，不影响其他槽位")

In [ ]:
# === 可视化：三种文件组织方式对比 ===

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# --- 1. 串行文件 Serial ---
ax = axes[0]
serial_data = ["Eve", "Bob", "Alice", "Diana", "Charlie"]
serial_times = ["9:15", "9:20", "9:25", "9:30", "9:35"]
colors_s = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for i, (name, t, c) in enumerate(zip(serial_data, serial_times, colors_s)):
    y = 4 - i
    rect = mpatches.FancyBboxPatch((0.5, y - 0.3), 3, 0.6, boxstyle="round,pad=0.05",
                                     facecolor=c, edgecolor='black', linewidth=1.5, alpha=0.8)
    ax.add_patch(rect)
    ax.text(2, y, f"{t}  {name}", ha='center', va='center', fontsize=12, 
            fontweight='bold', color='white')
    # 箭头表示搜索路径
    if i < len(serial_data) - 1:
        ax.annotate('', xy=(0.3, y - 0.4), xytext=(0.3, y - 0.05),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

ax.annotate('Search: top to bottom', xy=(0.3, 4.5), fontsize=10, color='gray',
            ha='center')
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-0.5, 5.2)
ax.set_title('串行文件 Serial File\n按到达时间存储，无序', fontsize=13, fontweight='bold')
ax.axis('off')

# --- 2. 顺序文件 Sequential ---
ax = axes[1]
seq_data = sorted(serial_data)
for i, name in enumerate(seq_data):
    y = 4 - i
    c = '#3498db'
    rect = mpatches.FancyBboxPatch((0.5, y - 0.3), 3, 0.6, boxstyle="round,pad=0.05",
                                     facecolor=c, edgecolor='black', linewidth=1.5, alpha=0.8)
    ax.add_patch(rect)
    ax.text(2, y, f"[{i}]  {name}", ha='center', va='center', fontsize=12,
            fontweight='bold', color='white')

# 二分查找箭头
ax.annotate('Binary\nSearch', xy=(4.2, 2), fontsize=11, color='#e74c3c',
            fontweight='bold', ha='center')
ax.annotate('', xy=(3.7, 2), xytext=(4.2, 2.8),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2))

ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 5.2)
ax.set_title('顺序文件 Sequential File\n按键值排序存储', fontsize=13, fontweight='bold')
ax.axis('off')

# --- 3. 随机访问文件 Random Access ---
ax = axes[2]
hash_positions = {"Alice": 2, "Bob": 5, "Charlie": 0, "Diana": 3, "Eve": 8}
table_cells = [None] * 10
for name, pos in hash_positions.items():
    table_cells[pos] = name

for i in range(10):
    y = 9 - i
    c = '#2ecc71' if table_cells[i] else '#ecf0f1'
    rect = mpatches.FancyBboxPatch((0.5, y*0.5 - 0.2), 3, 0.4, boxstyle="round,pad=0.03",
                                     facecolor=c, edgecolor='black', linewidth=1, alpha=0.8)
    ax.add_patch(rect)
    content = table_cells[i] if table_cells[i] else "<empty>"
    text_color = 'white' if table_cells[i] else 'gray'
    ax.text(2, y*0.5, f"[{i}] {content}", ha='center', va='center', fontsize=10,
            fontweight='bold', color=text_color)

# Hash arrow
ax.annotate('hash("Alice")\n= 2', xy=(0.3, 2*0.5 + 3.5), xytext=(-1.5, 2*0.5 + 3.5),
            fontsize=10, color='#e74c3c', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2),
            ha='center')

ax.set_xlim(-3, 4.5)
ax.set_ylim(-0.5, 5.5)
ax.set_title('随机访问文件 Random Access\n哈希直接定位', fontsize=13, fontweight='bold')
ax.axis('off')

plt.suptitle('三种文件组织方式对比 File Organisation Comparison', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === 可视化：哈希碰撞演示 Hash Collision Visualization ===

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# --- 开放寻址法 Open Addressing ---
ax = axes[0]
table_size = 7
items = [("Cat", 0), ("Dog", 1), ("Bat", 0), ("Ant", 3), ("Fox", 0)]
# 模拟线性探查
open_table = [None] * table_size
probe_paths = []

for key, hash_val in items:
    path = []
    idx = hash_val
    while open_table[idx] is not None:
        path.append((idx, 'collision'))
        idx = (idx + 1) % table_size
    path.append((idx, 'placed'))
    open_table[idx] = key
    probe_paths.append((key, hash_val, path))

# Draw table
for i in range(table_size):
    y = (table_size - 1 - i) * 0.8
    c = '#2ecc71' if open_table[i] else '#ecf0f1'
    rect = mpatches.FancyBboxPatch((2, y), 2, 0.6, boxstyle="round,pad=0.05",
                                     facecolor=c, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    content = open_table[i] if open_table[i] else "empty"
    ax.text(3, y + 0.3, f"[{i}] {content}", ha='center', va='center', fontsize=11,
            fontweight='bold', color='white' if open_table[i] else 'gray')

# Draw probe info
y_text = 5.5
for key, hash_val, path in probe_paths:
    probes_str = " -> ".join([f"[{p}]{'!' if s == 'collision' else '*'}" for p, s in path])
    collision = len(path) > 1
    color = '#e74c3c' if collision else '#27ae60'
    ax.text(0.1, y_text, f"Insert '{key}' (hash={hash_val}): {probes_str}",
            fontsize=9, color=color, fontweight='bold')
    y_text -= 0.4

ax.text(0.1, y_text - 0.3, "! = collision, * = placed", fontsize=9, color='gray')
ax.set_xlim(-0.5, 5)
ax.set_ylim(-0.5, 6.5)
ax.set_title('开放寻址法 Open Addressing\n(Linear Probing)', fontsize=13, fontweight='bold')
ax.axis('off')

# --- 链地址法 Separate Chaining ---
ax = axes[1]
chain_table = [[] for _ in range(table_size)]
for key, hash_val in items:
    chain_table[hash_val].append(key)

for i in range(table_size):
    y = (table_size - 1 - i) * 0.8
    has_data = len(chain_table[i]) > 0
    c = '#3498db' if has_data else '#ecf0f1'
    rect = mpatches.FancyBboxPatch((0.5, y), 1.2, 0.6, boxstyle="round,pad=0.05",
                                     facecolor=c, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(1.1, y + 0.3, f"[{i}]", ha='center', va='center', fontsize=11,
            fontweight='bold', color='white' if has_data else 'gray')
    
    # Draw chain
    for j, key in enumerate(chain_table[i]):
        x_chain = 2.2 + j * 1.5
        c_chain = '#e74c3c' if j > 0 else '#2ecc71'
        rect_chain = mpatches.FancyBboxPatch((x_chain, y), 1.2, 0.6, 
                                               boxstyle="round,pad=0.05",
                                               facecolor=c_chain, edgecolor='black', linewidth=1.5)
        ax.add_patch(rect_chain)
        ax.text(x_chain + 0.6, y + 0.3, key, ha='center', va='center', fontsize=10,
                fontweight='bold', color='white')
        # Arrow
        if j == 0:
            ax.annotate('', xy=(x_chain, y + 0.3), xytext=(1.7, y + 0.3),
                        arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
        else:
            ax.annotate('', xy=(x_chain, y + 0.3), xytext=(x_chain - 0.3, y + 0.3),
                        arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

ax.set_xlim(-0.2, 7)
ax.set_ylim(-0.5, 6.5)
ax.set_title('链地址法 Separate Chaining', fontsize=13, fontweight='bold')
ax.axis('off')

plt.suptitle('哈希碰撞处理方法对比 Collision Resolution Methods', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === 性能对比：三种组织方式的搜索时间 ===

def benchmark_search(n_records, n_searches=100):
    """比较三种方式的搜索性能"""
    # 生成数据
    records = [f"Record_{i:06d}" for i in range(n_records)]
    search_keys = [random.choice(records) for _ in range(n_searches)]
    
    # 串行 - 线性搜索
    serial_accesses = []
    for key in search_keys:
        for i, r in enumerate(records):
            if r == key:
                serial_accesses.append(i + 1)
                break
    
    # 顺序 - 二分搜索
    sorted_records = sorted(records)
    seq_accesses = []
    for key in search_keys:
        low, high = 0, len(sorted_records) - 1
        count = 0
        while low <= high:
            count += 1
            mid = (low + high) // 2
            if sorted_records[mid] == key:
                break
            elif sorted_records[mid] < key:
                low = mid + 1
            else:
                high = mid - 1
        seq_accesses.append(count)
    
    # 随机 - 哈希
    hash_dict = {r: i for i, r in enumerate(records)}
    hash_accesses = [1] * n_searches  # 理想情况O(1)
    
    return {
        'serial': np.mean(serial_accesses),
        'sequential': np.mean(seq_accesses),
        'random': np.mean(hash_accesses)
    }

# 不同数据量的测试
sizes = [100, 500, 1000, 5000, 10000]
results = {'serial': [], 'sequential': [], 'random': []}

print("=== 性能基准测试 Performance Benchmark ===")
print(f"{'Records':<10} {'Serial (avg)':<16} {'Sequential (avg)':<20} {'Random (avg)':<16}")
print("-" * 62)

for n in sizes:
    r = benchmark_search(n, 50)
    for method in results:
        results[method].append(r[method])
    print(f"{n:<10} {r['serial']:<16.1f} {r['sequential']:<20.1f} {r['random']:<16.1f}")

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sizes, results['serial'], 'o-', label='Serial (Linear)', color='#e74c3c', linewidth=2, markersize=8)
ax.plot(sizes, results['sequential'], 's-', label='Sequential (Binary)', color='#3498db', linewidth=2, markersize=8)
ax.plot(sizes, results['random'], '^-', label='Random (Hash)', color='#2ecc71', linewidth=2, markersize=8)

ax.set_xlabel('Number of Records', fontsize=12)
ax.set_ylabel('Average Accesses per Search', fontsize=12)
ax.set_title('Search Performance: Serial vs Sequential vs Random Access\n\u641c\u7d22\u6027\u80fd\u5bf9\u6bd4', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

---

## 四、深入理解：为什么哈希无处不在？

### 1. 数据库系统
- **索引 (Index)**：数据库使用哈希索引实现 O(1) 查找
- **连接操作 (JOIN)**：Hash Join 是最快的表连接算法之一

### 2. 操作系统
- **文件系统**：通过哈希快速定位文件的磁盘位置
- **页表 (Page Table)**：虚拟内存地址到物理地址的映射
- **进程调度**：通过PID哈希快速找到进程控制块

### 3. 网络与安全
- **密码存储**：不存储明文密码，只存储 hash(password)
- **数字签名**：用哈希验证文件完整性
- **区块链**：每个区块包含前一个区块的哈希值

### 4. 编程语言
- Python的 `dict` 和 `set` 底层就是哈希表！
- Java的 `HashMap`、C++的 `unordered_map` 都是哈希表

In [ ]:
# === 实际应用：密码哈希存储 ===

def hash_password(password):
    """使用SHA-256哈希密码（简化演示）"""
    return hashlib.sha256(password.encode()).hexdigest()

# 模拟密码存储
password_db = {}

def register(username, password):
    password_db[username] = hash_password(password)
    print(f"注册 '{username}': password hash = {password_db[username][:20]}...")

def login(username, password):
    if username in password_db:
        if password_db[username] == hash_password(password):
            return True
    return False

print("=== 密码哈希存储演示 Password Hashing ===")
register("alice", "MySecretPass123")
register("bob", "AnotherPass456")

print(f"\n尝试登录:")
print(f"  alice + 'MySecretPass123': {'Success' if login('alice', 'MySecretPass123') else 'Failed'}")
print(f"  alice + 'wrongpassword':   {'Success' if login('alice', 'wrongpassword') else 'Failed'}")

print(f"\n关键点：")
print(f"  1. 数据库中不存储明文密码，只存储哈希值")
print(f"  2. 相同密码总是产生相同哈希 -> 可以验证")
print(f"  3. 从哈希值无法反推出密码 -> 即使数据库泄露也安全")
print(f"  4. 微小改动产生完全不同的哈希:")
print(f"     'password' -> {hash_password('password')[:30]}...")
print(f"     'Password' -> {hash_password('Password')[:30]}...")

In [ ]:
# === 可视化：哈希表装载因子与碰撞关系 ===

def simulate_collisions(table_size, n_items):
    """模拟不同装载因子下的碰撞次数"""
    table = [False] * table_size
    collisions = 0
    for i in range(n_items):
        idx = random.randint(0, table_size - 1)
        if table[idx]:
            collisions += 1
        table[idx] = True
    return collisions

table_size = 100
load_factors = np.arange(0.1, 1.01, 0.05)
collision_rates = []

for lf in load_factors:
    n_items = int(table_size * lf)
    # 多次模拟取平均
    avg_collisions = np.mean([simulate_collisions(table_size, n_items) for _ in range(100)])
    collision_rates.append(avg_collisions / max(n_items, 1) * 100)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(load_factors * 100, collision_rates, 'o-', color='#e74c3c', linewidth=2, markersize=6)
ax.fill_between(load_factors * 100, collision_rates, alpha=0.2, color='#e74c3c')

# 标注关键区域
ax.axvline(x=70, color='#f39c12', linestyle='--', linewidth=2, label='Recommended max (70%)')
ax.axvline(x=50, color='#2ecc71', linestyle='--', linewidth=2, label='Ideal range (50%)')

ax.set_xlabel('Load Factor (%)', fontsize=12)
ax.set_ylabel('Collision Rate (%)', fontsize=12)
ax.set_title('Hash Table: Load Factor vs Collision Rate\n\u88c5\u8f7d\u56e0\u5b50\u4e0e\u78b0\u649e\u7387\u7684\u5173\u7cfb', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

ax.annotate('Danger Zone!\nToo many collisions', 
            xy=(90, collision_rates[-3]), fontsize=11, color='#e74c3c',
            fontweight='bold', ha='center',
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2),
            xytext=(75, collision_rates[-3] + 10))

plt.tight_layout()
plt.show()

print("结论：装载因子越高，碰撞越多，性能越差")
print("建议：保持装载因子在50%-70%之间")
print("超过70%时应该扩容（rehashing）")

---

## 五、三种方式对比总结

| 特征 | 串行 Serial | 顺序 Sequential | 随机 Random |
|:---|:---|:---|:---|
| **存储顺序** | 到达顺序 | 按键排序 | 哈希决定 |
| **搜索方法** | 线性搜索 | 二分搜索 | 哈希计算 |
| **搜索复杂度** | O(n) | O(log n) | O(1) 平均 |
| **插入复杂度** | O(1) 追加 | O(n) 需移动 | O(1) 平均 |
| **适用场景** | 日志、临时数据 | 报表、范围查询 | 数据库、字典 |
| **空间利用** | 高（无浪费） | 高 | 中（需预留空间） |
| **范围查询** | 差 | **优秀** | 差 |

---

## 练习题 Exercises

### 练习1：概念题

1. 比较串行文件和顺序文件的区别，各给出一个适用场景。(4分)
2. 解释哈希表中"碰撞"是什么，为什么一定会发生？(3分)
3. 比较开放寻址法和链地址法的优缺点。(4分)
4. 解释为什么哈希表的大小通常选择质数。(2分)
5. 解释"装载因子"是什么，为什么它影响哈希表性能。(3分)

### 练习2：编程题

In [ ]:
# 练习2a：实现一个完整的哈希表，包含：
# - insert(key, value)
# - search(key) -> value
# - delete(key)
# - 使用二次探查法（quadratic probing）处理碰撞
#   即碰撞后尝试 index+1^2, index+2^2, index+3^2, ...

# 你的代码：


In [ ]:
# 练习2b：实现一个简单的文件索引系统
# - 用顺序文件存储学生记录（按学号排序）
# - 用哈希表建立"姓名 -> 学号"的索引
# - 实现按学号搜索（二分查找）和按姓名搜索（哈希查找）

# 你的代码：


In [ ]:
# 练习2c：测试不同哈希函数的碰撞率
# 定义至少3种不同的哈希函数
# 对1000个随机字符串进行哈希（表大小=101）
# 用柱状图可视化每种哈希函数的碰撞次数

# 你的代码：


### 练习3：考试题风格

**Question 1** (Cambridge style)

A company stores employee records using a hash table with 1000 slots.
The hash function is: `hash(EmployeeID) = EmployeeID MOD 1000`

(a) Calculate the hash values for employees with IDs: 2345, 7890, 1345. [3]

(b) Explain what happens when two employees have the same hash value. [2]

(c) Describe how linear probing would resolve this collision. [3]

(d) Explain one advantage and one disadvantage of using random file organisation compared to sequential file organisation for this data. [4]

---

### 本节小结 Summary

- **串行文件**：按到达顺序存储，写入O(1)，搜索O(n)
- **顺序文件**：按键排序存储，支持二分搜索O(log n)
- **随机访问文件**：通过哈希函数直接定位，平均O(1)
- **碰撞处理**：开放寻址法（在表内探查空位）vs 链地址法（用链表存储碰撞元素）
- 哈希是计算机科学的核心技术，广泛应用于数据库、操作系统、安全等领域